# Modelo Preditivo de contas a receber  (Gradient Boosting)

### Objetivo:
Nessa etapa, o objetivo é desenvolver um modelo preditivo para estimar o valor dos recebíveis em determinado período. O sue objetivo é ajudar as empresas a preverem seus fluxos de caixa futuros com maior precisão, permitindo um melhor planejamento e evitando déficits devido a inadimplências inesperadas.

### Abordagem:
Para isso, será utilizado um modelo de classificação, como Gradient Boosting para prever os pagamentos inadimplentes. A partir disso, todos os pagamentos classificados como inadimplentes serão desconsiderados na soma dos recebíveis previstos. 



## Vantagens do Gradient Boosting
- Alta performance: costuma apresentar resultados superiores a modelos lineares em bases tabulares  
- Flexibilidade: permite ajustar diversos hiperparâmetros para controlar viés e variância  
- Regularização: parâmetros como learning rate, subsample e min_samples_leaf ajudam a evitar overfitting  
- Interpretação relativa: oferece feature importance para entender quais variáveis mais impactam o modelo  
- Compatibilidade: integrado diretamente ao scikit-learn, facilitando o uso de ferramentas como GridSearchCV e cross-validation  

## Sumário do Notebook
1. Instalação de dependências – garantir que bibliotecas essenciais estão disponíveis  
2. Carregamento dos dados – importação e leitura da base de cobranças  
3. Pré-processamento – limpeza, padronização e transformação dos dados  
4. Engenharia de atributos – criação de features comportamentais e históricas  
5. Treinamento do modelo Gradient Boosting – ajuste inicial e explicação dos parâmetros principais  
6. Otimização de hiperparâmetros com GridSearchCV – busca sistemática pelos melhores parâmetros com validação cruzada  
7. Avaliação do modelo otimizado – métricas de desempenho no conjunto de teste
8. Previsão de Recebíveis em Período Específico - aplicação do modelo para estimar valores futuros considerando inadimplência prevista

## Tecnologias Utilizadas
- pandas – manipulação e análise de dados financeiros  
- numpy – operações matemáticas e vetorização de cálculos  
- matplotlib.pyplot e seaborn – visualização do comportamento de pagamentos, distribuição de scores e erros do modelo  
- scikit-learn – ferramentas para divisão de dados (train_test_split), ajuste de hiperparâmetros (GridSearchCV), validação cruzada e métricas de classificação  



### **1. Instalando Dependências**




1. **Garantia da biblioteca `openpyxl`**  
   O bloco inicial verifica se a biblioteca `openpyxl` está instalada.  
   - Caso esteja ausente, o código utiliza o módulo `subprocess` para executar um comando `pip install` dentro do próprio ambiente Python, garantindo que arquivos no formato `.xlsx` possam ser lidos pelo `pandas`.  
   
   - O parâmetro `check=False` impede que uma falha na instalação interrompa a execução do notebook.

2. **Importação de bibliotecas essenciais**  
   - `pandas` e `numpy`: utilizados para manipulação e análise de dados, leitura de planilhas, criação e transformação de DataFrames, além de cálculos numéricos eficientes.  

   - `scikit-learn`: fornece ferramentas para dividir dados em treino e teste (`train_test_split`), realizar otimização de hiperparâmetros (`GridSearchCV`), executar validação cruzada (`cross_val_score`) e avaliar modelos com métricas de classificação.  

   - `GradientBoostingClassifier`: o algoritmo de aprendizado de máquina que será usado para construir o modelo preditivo de score de crédito.  

   - Métricas como `accuracy_score`, `precision_score`, `recall_score`, `f1_score` e `classification_report` serão aplicadas para avaliar o desempenho do modelo.




In [24]:
import sys
# Garante openpyxl para leitura de .xlsx (sem interromper caso já exista)
try:
    import openpyxl  # noqa: F401
except Exception:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl"], check=False)

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)
from sklearn.ensemble import GradientBoostingClassifier

print("Passo 1: Carregando e Estruturando os dados, importando as bibliotecas necessárias")



Passo 1: Carregando e Estruturando os dados, importando as bibliotecas necessárias


### **2. Leitura e carregamento dos Dados**



**2.1.** **Definição dos arquivos**  
   A lista `arquivos_excel` contém os caminhos para todos os arquivos que serão lidos e unificados.

**2.2.** **Mapeamento de colunas**  
   O dicionário `mapa_de_colunas` traduz nomes originais (como `data_vencto` ou `vl_boleto`) para um formato padronizado (`data_vencimento`, `valor_original`, etc.), garantindo consistência nos nomes durante o pré-processamento.

**2.3.** **Leitura e padronização**  
   O loop percorre cada arquivo da lista, tenta:
   - Ler o conteúdo com `pandas.read_excel`.  
   - Converter todos os nomes de colunas para letras minúsculas (`str.lower`).  
   - Renomear as colunas de acordo com o mapeamento definido (`rename`).  
   - Adicionar o DataFrame padronizado à lista `lista_de_dfs`.  

   Caso o arquivo não seja encontrado ou ocorra algum erro durante a leitura, mensagens informativas são exibidas.

**2.4.** **Unificação dos dados**  
   Todos os DataFrames lidos são concatenados em um único DataFrame `df` usando `pd.concat`.  
   - Caso nenhum arquivo seja carregado, cria-se um DataFrame vazio para evitar erros.  
   - Valores literais `\N` (indicando dados ausentes) são substituídos por `NaN` para facilitar o tratamento posterior.

**2.5.** **Verificação inicial**  
   O código imprime o total de linhas unificadas e exibe as cinco primeiras linhas do DataFrame final para confirmar que os dados foram carregados corretamente.


In [3]:
# Caminhos dos arquivos para serem lidos
arquivos_excel = [
    "../dados/Grupo1-GL.xlsx",
    "../dados/Grupo3-GP.xlsx",
    "../dados/Grupo4-GT.xlsx",
    "../dados/Grupo2-GM.xlsx"
]

# Padronização dos nomes das colunas
mapa_de_colunas = {
    'data_vencto': 'data_vencimento',
    'vl_boleto': 'valor_original',
    'dt_pagto': 'data_pagamento',
    'vl_pagto': 'valor_pago',
    'id_pagador': 'pagador',
    'banco_emissor': 'banco',
    'qtd_acessos_pagador': 'qtde_acessado_pagador',
    'teve_acesso_pagador': 'teve_acesso_pagador',
    'pagador_cep': 'pagador_cep',
    'pagador_cidade': 'pagador_cidade',
    'status_boleto': 'status_boleto',
    'data_inclusao': 'data_inclusao',
    'grupo': 'grupo',
    'empresa': 'empresa'
}

lista_de_dfs = []
print("\nLendo e padronizando os arquivos...")

# Lê cada arquivo Excel da lista, padroniza os nomes das colunas e junta todos em uma unica lista de DataFrames. 
for arquivo in arquivos_excel:
    try:
        df_temp = pd.read_excel(arquivo)              
        df_temp.columns = df_temp.columns.str.lower()  # uniformiza nomes para minúsculo
        df_temp.rename(columns=mapa_de_colunas, inplace=True)  # aplica mapeamento
        lista_de_dfs.append(df_temp)
        print(f"  - '{arquivo}' carregado e padronizado.")
    except FileNotFoundError:
        print(f"  - Arquivo não encontrado: '{arquivo}'.")
    except Exception as e:
        print(f"  - Erro ao processar '{arquivo}': {e}")

# Concatenação dos dfs lidos
df = pd.concat(lista_de_dfs, ignore_index=True) if lista_de_dfs else pd.DataFrame()

print("\nArquivos unificados.")
print(f"Total de linhas: {len(df)}")

if not df.empty:
    df.replace('\\N', np.nan, inplace=True)  # converte strings '\N' em valores nulos para ajudar no tratamento
    print("\nPré-visualização dos dados unificados:")
    display(df.head())
else:
    print("\nNenhum dado foi carregado. Verifique os caminhos dos arquivos.")



Lendo e padronizando os arquivos...
  - Arquivo '../dados/Grupo1-GL.xlsx' carregado e padronizado com sucesso.
  - Arquivo '../dados/Grupo3-GP.xlsx' carregado e padronizado com sucesso.
  - Arquivo '../dados/Grupo4-GT.xlsx' carregado e padronizado com sucesso.
  - Arquivo '../dados/Grupo2-GM.xlsx' carregado e padronizado com sucesso.

Arquivos unificados com sucesso!
Total de linhas carregadas de todos os arquivos: 1202864


/var/folders/f4/9yt_2mc11_j28v6p0byhbk580000gn/T/ipykernel_4227/2804603164.py:49: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('\\N', np.nan, inplace=True)



Primeiras 5 linhas do DataFrame UNIFICADO:


,id_grupo,id_beneficiario,numero_do_boleto,data_inclusao,status_boleto,data_vencimento,valor_original,data_pagamento,valor_pago,banco,...,pagador_cidade,qtde_acessado_pagador,pagador_dt_ultimo_acesso,pagador_cnpjcpf,pagador_inscricao_hash,valor_abatimento,tipo_juros,juros,tipo_multa,multa
0,173,554,426309011,2024-11-25,REGISTRADO,2025-12-03 00:00:00,188645.00,NaT,NaN,ITAU UNIBANCO S.A.,...,CAMPO GRANDE,NaN,NaT,CNPJ,dc1e3c141c9c7d7408b783c9d4f48bc3721dd9621e4fd4...,0.0,M,62.88,P,2.0
1,173,554,426702880,2024-11-26,REGISTRADO,2025-12-03 00:00:00,69386.15,NaT,NaN,ITAU UNIBANCO S.A.,...,CAMPO GRANDE,NaN,NaT,CNPJ,dc1e3c141c9c7d7408b783c9d4f48bc3721dd9621e4fd4...,0.0,M,23.13,P,2.0
2,173,554,427422093,2024-11-28,REGISTRADO,2025-12-03 00:00:00,21717.33,NaT,NaN,ITAU UNIBANCO S.A.,...,CAMPO GRANDE,NaN,NaT,CNPJ,dc1e3c141c9c7d7408b783c9d4f48bc3721dd9621e4fd4...,0.0,M,7.24,P,2.0
3,173,554,439264582,2025-01-21,REGISTRADO,2025-05-12 00:00:00,221338.95,NaT,NaN,ITAU UNIBANCO S.A.,...,BRASILIA,NaN,NaT,CNPJ,04ffcdcdbb6d616c87ab1c9a5e59586dbc46f95c0e4e98...,0.0,M,73.78,P,2.0
4,173,554,439264583,2025-01-21,REGISTRADO,2025-05-12 00:00:00,110669.49,NaT,NaN,ITAU UNIBANCO S.A.,...,BRASILIA,NaN,NaT,CNPJ,04ffcdcdbb6d616c87ab1c9a5e59586dbc46f95c0e4e98...,0.0,M,36.89,P,2.0


## **3. Pré-processamento dos Dados**

### Pré-processamento dos Dados

Nesta etapa, o conjunto de dados unificado passa por limpeza, conversão de tipos e criação de novas variáveis necessárias para o modelo de previsão.

**3.1.** **Mensagem inicial e contagem de linhas**  
   O código imprime o número de linhas existentes antes do início da limpeza, permitindo acompanhar a redução após o processo.

**3.2.** **Conversão de tipos**  
   - `data_vencimento`, `data_pagamento` e `data_inclusao` são convertidas para o tipo `datetime`, garantindo consistência em análises temporais.  
   - `valor_original` é convertido para tipo numérico para possibilitar cálculos financeiros.  
   - `qtde_acessado_pagador` também é convertida para numérico; valores inválidos são tratados como `0`.

**3.3.** **Remoção de linhas incompletas**  
   São eliminadas linhas que não possuem informações essenciais como data de vencimento, valor original ou identificador do pagador.  
   Ao final, é exibido quantas linhas foram removidas, quantas restaram e o percentual de dados válidos retidos.

**3.4.** **Cálculo de dias de atraso**  
   Cria-se a coluna `dias_de_atraso` a partir da diferença entre `data_pagamento` e `data_vencimento`.

**3.5.** **Definição do status de pagamento**  
   A função `definir_status` classifica cada registro em:
   - *Em Dia* (pagamento até 1 dia após o vencimento)  
   - *Atraso* (pagamento entre 2 e 30 dias após o vencimento)  
   - *Inadimplente* (pagamento ausente ou atraso superior a 30 dias)  

   Essa informação é armazenada na nova coluna `status_pagamento`.

**3.6.** **Relatório de pré-processamento**  
   É exibida a distribuição de registros em cada status de pagamento e uma pré-visualização das colunas principais após o processamento.

**3.7.** **Tratamento de acesso do pagador**  
   - Garante que a coluna `qtde_acessado_pagador` está em formato numérico.  
   - Caso a coluna `teve_acesso_pagador` não exista, ela é criada como variável binária:  
     `1` quando houve pelo menos um acesso do pagador e `0` caso contrário.  
   - Ao final, é mostrado o resumo da distribuição dessa variável e alguns exemplos de seus valores.

Essa etapa garante que os dados estejam limpos, tipados corretamente e com variáveis derivadas relevantes para análise e modelagem.


In [ ]:
print("Passo 2: Pré Processamento dos dados")

linhas_iniciais = len(df)
print(f"Número de linhas antes da limpeza: {linhas_iniciais}")

print("\nIniciando conversão de tipos e limpeza...")

# Tipos
df['data_vencimento'] = pd.to_datetime(df['data_vencimento'], format='%Y-%m-%d', errors='coerce')
df['data_pagamento']  = pd.to_datetime(df['data_pagamento'],  format='%Y-%m-%d', errors='coerce')
df['data_inclusao']   = pd.to_datetime(df['data_inclusao'],   errors='coerce')
df['valor_original']  = pd.to_numeric(df['valor_original'], errors='coerce')
df['qtde_acessado_pagador'] = pd.to_numeric(df['qtde_acessado_pagador'], errors='coerce').fillna(0)

# Linhas essenciais
df.dropna(subset=['data_vencimento', 'valor_original', 'pagador'], inplace=True)

linhas_finais = len(df)
print(f"Limpeza concluída. Foram removidas {linhas_iniciais - linhas_finais} linhas com dados essenciais ausentes.")
print(f"Número de linhas após a limpeza: {linhas_finais}")
print(f"Percentual de dados válidos retidos: {((linhas_finais / max(linhas_iniciais,1)) * 100):.2f}%")

# Variáveis de atraso e status
df['dias_de_atraso'] = (df['data_pagamento'] - df['data_vencimento']).dt.days

def definir_status(dias):
    if pd.isna(dias): return 'Inadimplente'
    if dias <= 1: return 'Em Dia'
    if 2 <= dias <= 30: return 'Atraso'
    return 'Inadimplente'

df['status_pagamento'] = df['dias_de_atraso'].apply(definir_status)

print("\nRelatório do Pré-processamento — Distribuição do Status de Pagamento:")
print(df['status_pagamento'].value_counts(dropna=False))
display(df[['data_vencimento','data_pagamento','dias_de_atraso','status_pagamento','valor_original','pagador']].head())

# Garantir numérica 
df['qtde_acessado_pagador'] = pd.to_numeric(df.get('qtde_acessado_pagador'), errors='coerce').fillna(0)

# Criar teve_acesso_pagador se não existir (binária 0/1 a partir da contagem)
if 'teve_acesso_pagador' not in df.columns:
    df['teve_acesso_pagador'] = (df['qtde_acessado_pagador'] > 0).astype(int)

print("Resumo teve_acesso_pagador:")
print(df['teve_acesso_pagador'].value_counts(dropna=False))
print(df[['qtde_acessado_pagador','teve_acesso_pagador']].head())



Passo 2: Pré Processamento dos dados
Número de linhas antes da limpeza: 1172662

Iniciando conversão de tipos e limpeza...
Limpeza concluída. Foram removidas 0 linhas com dados essenciais ausentes.
Número de linhas após a limpeza: 1172662
Percentual de dados válidos retidos: 100.00%

Relatório do Pré-processamento — Distribuição do Status de Pagamento:
status_pagamento
Inadimplente    521308
Em Dia          458497
Atraso          192857
Name: count, dtype: int64


,data_vencimento,data_pagamento,dias_de_atraso,status_pagamento,valor_original,pagador
514626,2024-09-25,2024-09-24,-1.0,Em Dia,330.85,1066354
514627,2024-10-25,2024-10-02,-23.0,Em Dia,330.85,1066354
534904,2024-10-25,2024-10-09,-16.0,Em Dia,1239.76,1066354
514628,2024-11-25,2024-11-05,-20.0,Em Dia,330.85,1066354
534905,2024-11-25,2024-11-05,-20.0,Em Dia,1239.76,1066354


Resumo teve_acesso_pagador:
teve_acesso_pagador
0    660658
1    512004
Name: count, dtype: int64
        qtde_acessado_pagador  teve_acesso_pagador
514626                    9.0                    1
514627                    1.0                    1
534904                    4.0                    1
514628                    1.0                    1
534905                    1.0                    1


## **4. Feature Engineering**
Nesta etapa são criadas novas variáveis (*features*) que enriquecem o conjunto de dados com informações derivadas, históricas e comportamentais dos pagadores, ajudando o modelo a capturar padrões relevantes para prever inadimplência.

**4.1.** **Datas derivadas**  
   - `mes_vencimento`: mês da data de vencimento.  
   - `dia_mes_vencimento`: dia do mês do vencimento.  
   - `dia_semana_vencimento`: dia da semana (0 = segunda, 6 = domingo).

**4.2.** **Estatísticas financeiras por pagador**  
   - `valor_medio_pagador`: média do valor de boletos já emitidos para cada pagador.  
   - `razao_valor_vs_media`: razão entre o valor atual e a média histórica do pagador (limitada a 10 para evitar outliers).  
   - `hist_std_dias_atraso`: desvio padrão do atraso histórico do pagador.

**4.3.** **Tempo de relacionamento**  
   - `tempo_relacionamento_dias`: número de dias entre a primeira cobrança registrada do pagador e o vencimento atual.

**4.4.** **Tipo de pagador**  
   - `tipo_pagador`: utiliza a coluna `pagador_cnpjcpf` se existir, caso contrário atribui "N/A".

**4.5.** **Intervalo emissão → vencimento**  
   - `dias_emissao_vencimento`: dias entre a inclusão do boleto e seu vencimento.

**4.6.** **Histórico de pagamentos e anti-*data leak***  
   - Os registros são ordenados por `pagador` e `data_vencimento`.  
   - `status_pagamento_anterior`: status do pagamento anterior do mesmo pagador (ou "Novo Cliente" se não houver histórico).  
   - `hist_total_cobrancas`: total de cobranças registradas por pagador.  
   - `hist_media_dias_atraso`: média de dias de atraso do pagador.  
   - `hist_taxa_inadimplencia`: proporção de boletos anteriores inadimplentes.  
   - `hist_taxa_atraso`: proporção de boletos anteriores pagos com atraso.

**4.7.** **Interações e indicadores de risco**  
   - `vencimento_prox_fds`: indica se o vencimento cai próximo ao fim de semana (quinta ou sexta-feira).  
   - `risco_interacao_valor_atraso`: combina a razão do valor atual com a taxa histórica de atraso do pagador.

**4.8.** **Variável alvo (target)**  
   - `target_inad`: variável binária que marca boletos inadimplentes (1 = inadimplente, 0 = pago ou em dia).

Após essa etapa, o conjunto de dados passa a ter uma base mais rica e informativa, incluindo variáveis temporais, comportamentais e históricas, que aumentam a capacidade do modelo de identificar padrões de inadimplência.


In [ ]:
# Datas derivadas usadas no modelo
df['mes_vencimento']       = df['data_vencimento'].dt.month
df['dia_mes_vencimento']   = df['data_vencimento'].dt.day
df['dia_semana_vencimento']= df['data_vencimento'].dt.dayofweek

# Estatísticas por pagador
df['valor_medio_pagador']  = df.groupby('pagador')['valor_original'].transform('mean')
df['razao_valor_vs_media'] = (df['valor_original'] / df['valor_medio_pagador']).fillna(1).clip(0, 10)
df['hist_std_dias_atraso'] = df.groupby('pagador')['dias_de_atraso'].transform('std').fillna(0)

# Tempo de relacionamento
df['tempo_relacionamento_dias'] = (
    df['data_vencimento'] - df.groupby('pagador')['data_inclusao'].transform('min')
).dt.days.fillna(0)

# Tipo de pagador, se existir a coluna; senão, marcador neutro
if 'pagador_cnpjcpf' in df.columns:
    df['tipo_pagador'] = df['pagador_cnpjcpf']
else:
    df['tipo_pagador'] = 'N/A'

# Janela emissão → vencimento
df['dias_emissao_vencimento'] = (df['data_vencimento'] - df['data_inclusao']).dt.days.fillna(30)

# Anti-data-leak: ordenar e criar status anterior por pagador
df.sort_values(by=['pagador', 'data_vencimento'], inplace=True)
df['status_pagamento_anterior'] = df.groupby('pagador')['status_pagamento'].shift(1).fillna('Novo Cliente')

# Históricos agregados
df['hist_total_cobrancas']   = df.groupby('pagador')['pagador'].transform('count')
df['hist_media_dias_atraso'] = df.groupby('pagador')['dias_de_atraso'].transform(lambda x: x.fillna(0).mean())
df['hist_taxa_inadimplencia']= df.groupby('pagador')['status_pagamento'].transform(lambda x: (x == 'Inadimplente').mean())
df['hist_taxa_atraso']       = df.groupby('pagador')['status_pagamento'].transform(lambda x: (x == 'Atraso').mean())

# Interações úteis
df['vencimento_prox_fds'] = df['dia_semana_vencimento'].isin([3, 4]).astype(int)
df['risco_interacao_valor_atraso'] = df['razao_valor_vs_media'] * df['hist_taxa_atraso']

# Target
df['target_inad'] = (df['status_pagamento'] == 'Inadimplente').astype(int)

print("\nFeatures avançadas criadas com sucesso!")
display(
    df[
        ['valor_original','mes_vencimento','dia_semana_vencimento','dia_mes_vencimento',
         'valor_medio_pagador','qtde_acessado_pagador','teve_acesso_pagador',
         'hist_total_cobrancas','hist_media_dias_atraso','hist_taxa_inadimplencia',
         'razao_valor_vs_media','hist_std_dias_atraso','tempo_relacionamento_dias',
         'tipo_pagador','dias_emissao_vencimento','status_pagamento_anterior',
         'vencimento_prox_fds','hist_taxa_atraso','risco_interacao_valor_atraso',
         'data_vencimento','target_inad']
    ].head()
)


--- Iniciando a criação de features avançadas de comportamento ---

Features avançadas criadas com sucesso!


,valor_original,mes_vencimento,dia_semana_vencimento,dia_mes_vencimento,valor_medio_pagador,qtde_acessado_pagador,teve_acesso_pagador,hist_total_cobrancas,hist_media_dias_atraso,hist_taxa_inadimplencia,...,hist_std_dias_atraso,tempo_relacionamento_dias,tipo_pagador,dias_emissao_vencimento,status_pagamento_anterior,vencimento_prox_fds,hist_taxa_atraso,risco_interacao_valor_atraso,data_vencimento,target_inad
514626,330.85,9,2,25,785.305,9.0,1,30,-13.5,0.233333,...,5.960012,7,CPF,7,Novo Cliente,0,0.0,0.0,2024-09-25,0
514627,330.85,10,4,25,785.305,1.0,1,30,-13.5,0.233333,...,5.960012,37,CPF,37,Em Dia,1,0.0,0.0,2024-10-25,0
534904,1239.76,10,4,25,785.305,4.0,1,30,-13.5,0.233333,...,5.960012,37,CPF,21,Em Dia,1,0.0,0.0,2024-10-25,0
514628,330.85,11,0,25,785.305,1.0,1,30,-13.5,0.233333,...,5.960012,68,CPF,68,Em Dia,0,0.0,0.0,2024-11-25,0
534905,1239.76,11,0,25,785.305,1.0,1,30,-13.5,0.233333,...,5.960012,68,CPF,52,Em Dia,0,0.0,0.0,2024-11-25,0


## **5. seleção e Treinamento do Modelo Preditivo (Gradient Boosting)**

#### Preparação do Conjunto de Modelagem

Esta seção define o conjunto de variáveis de entrada e o alvo, garante a consistência das colunas, realiza a codificação das variáveis categóricas e separa os dados em treino e teste.

#### 5.1 Seleção de variáveis
- `features_final`: lista das colunas candidatas para o modelo  
- `target_col`: coluna de destino `target_inad`

#### 5.2 Validação das colunas e filtragem do DataFrame
- Cria `colunas_existentes` mantendo apenas as features que realmente estão presentes em `df`  
- Monta `df_modelo_ultra_final` com as colunas válidas mais o alvo e remove linhas com valores ausentes

#### 5.3 Matriz de atributos e vetor alvo
- `X_ultra_final`: cópia das features selecionadas  
- `y_ultra_final`: cópia da coluna alvo

#### 5.4 Codificação de variáveis categóricas
- Identifica as colunas categóricas disponíveis em `X_ultra_final` entre `banco`, `tipo_pagador`, `teve_acesso_pagador`, `status_pagamento_anterior`  
- Aplica `pd.get_dummies` com `drop_first=True` para gerar variáveis indicadoras e evitar multicolinearidade perfeita  
- Exibe uma amostra de `X_ultra_final` após a codificação

#### 5.5 Divisão em treino e teste
- Usa `train_test_split` com `test_size=0.25`, `random_state=42` e `stratify=y_ultra_final` para preservar a proporção de classes  
- Imprime as dimensões finais de `X_train_uf`, `X_test_uf`, `y_train_uf` e `y_test_uf`


In [ ]:

features_final = [
    'valor_original', 'mes_vencimento', 'dia_semana_vencimento', 
    'dia_mes_vencimento', 'valor_medio_pagador', 'banco',
    'qtde_acessado_pagador', 'teve_acesso_pagador',
    'hist_total_cobrancas', 'hist_media_dias_atraso', 'hist_taxa_inadimplencia',
    'razao_valor_vs_media', 'hist_std_dias_atraso', 'tempo_relacionamento_dias',
    'tipo_pagador', 'dias_emissao_vencimento', 'status_pagamento_anterior', 
    'vencimento_prox_fds', 'hist_taxa_atraso',
    'risco_interacao_valor_atraso'
]
target_col = 'target_inad'

# Garante que só usa colunas existentes
colunas_existentes = [col for col in features_final if col in df.columns]
df_modelo_ultra_final = df[colunas_existentes + [target_col]].dropna()

X_ultra_final = df_modelo_ultra_final[colunas_existentes].copy()
y_ultra_final = df_modelo_ultra_final[target_col].copy()

# Dummies nas categóricas que existem no X
colunas_categoricas = [c for c in ['banco','tipo_pagador','teve_acesso_pagador','status_pagamento_anterior'] if c in X_ultra_final.columns]
X_ultra_final = pd.get_dummies(X_ultra_final, columns=colunas_categoricas, drop_first=True)

print("Amostra das features após get_dummies:")
display(X_ultra_final.head())

# Split (mantém nomes)
X_train_uf, X_test_uf, y_train_uf, y_test_uf = train_test_split(
    X_ultra_final, y_ultra_final, test_size=0.25, random_state=42, stratify=y_ultra_final
)

print(f"Shapes — X_train_uf: {X_train_uf.shape}, X_test_uf: {X_test_uf.shape}")


Amostra das features após get_dummies:


,valor_original,mes_vencimento,dia_semana_vencimento,dia_mes_vencimento,valor_medio_pagador,qtde_acessado_pagador,hist_total_cobrancas,hist_media_dias_atraso,hist_taxa_inadimplencia,razao_valor_vs_media,...,banco_BANCO DAYCOVAL,banco_BANCO DO BRASIL,banco_BANCO SAFRA,banco_BANCO SANTANDER (BRASIL) S.A.,banco_ITAU UNIBANCO S.A.,tipo_pagador_CPF,teve_acesso_pagador_1,status_pagamento_anterior_Em Dia,status_pagamento_anterior_Inadimplente,status_pagamento_anterior_Novo Cliente
514626,330.85,9,2,25,785.305,9.0,30,-13.5,0.233333,0.421301,...,False,False,False,False,False,True,True,False,False,True
514627,330.85,10,4,25,785.305,1.0,30,-13.5,0.233333,0.421301,...,False,False,False,False,False,True,True,True,False,False
534904,1239.76,10,4,25,785.305,4.0,30,-13.5,0.233333,1.578699,...,False,False,False,False,False,True,True,True,False,False
514628,330.85,11,0,25,785.305,1.0,30,-13.5,0.233333,0.421301,...,False,False,False,False,False,True,True,True,False,False
534905,1239.76,11,0,25,785.305,1.0,30,-13.5,0.233333,1.578699,...,False,False,False,False,False,True,True,True,False,False


Shapes — X_train_uf: (879496, 29), X_test_uf: (293166, 29)


## **6. Otimização de Hiperparâmetros com GridSearchCV**


Nesta etapa é utilizado o GridSearchCV para encontrar a melhor combinação de parâmetros do modelo Gradient Boosting Classifier.  
A busca sistemática testa todas as combinações definidas em `param_grid`, aplicando validação cruzada com 3 folds e avaliando cada modelo pela métrica F1, que equilibra precisão e recall para problemas binários de classificação.


#### 6.1 Definição do espaço de busca
Essa etapa tem como objetivo definir o espaço de busca para os hiperparâmetros do modelo Gradient Boosting Classifier. A escolha cuidadosa desses parâmetros é crucial para otimizar o desempenho do modelo.

- `n_estimators` – número de árvores construídas sequencialmente no boosting. Mais árvores podem melhorar o ajuste, mas aumentam o tempo de treino e o risco de overfitting.  
- `learning_rate` – peso aplicado a cada árvore adicionada. Valores menores (como 0.05) tornam o aprendizado mais lento, mas ajudam a generalizar melhor.  
- `max_depth` – profundidade máxima das árvores. Controla a complexidade do modelo; valores maiores capturam padrões mais complexos, mas podem overfitar.  
- `min_samples_leaf` – número mínimo de amostras que cada folha precisa conter. Valores maiores simplificam as árvores e reduzem overfitting.  
- `min_samples_split` – número mínimo de amostras para que um nó interno seja dividido. Evita divisões com poucos dados e ajuda na generalização.  
- `max_features` – fração de variáveis consideradas em cada divisão. `"sqrt"` usa a raiz quadrada do total de features, trazendo aleatoriedade e ajudando a evitar overfitting.  
- `subsample` – porcentagem de amostras usadas para treinar cada árvore. Valores menores que 1 tornam o treino estocástico, aceleram e reduzem overfitting; 1.0 usa todo o conjunto.  


#### 6.2 Execução
O processo:
- Treinou e avaliou 288 modelos (96 combinações × 3 folds).  
- Rodou em paralelo usando todos os núcleos disponíveis (n_jobs=-1).  
- Registrou o progresso detalhado (verbose=2).

#### 6.3 Melhor configuração encontrada
Após testar todas as combinações, o GridSearchCV selecionou os seguintes hiperparâmetros como ideais para maximizar a métrica F1:

```python
{
    'learning_rate': 0.1,
    'max_depth': 5,
    'max_features': 'sqrt',
    'min_samples_leaf': 50,
    'min_samples_split': 10,
    'n_estimators': 200,
    'subsample': 1.0
}


In [ ]:
# Define o espaço de busca de hiperparâmetros que o GridSearchCV vai testar.
# Aqui são ajustados parâmetros importantes do Gradient Boosting, como número de árvores, profundidade máxima, tamanho mínimo de folhas e taxa de aprendizado.
param_grid = {
    "n_estimators":   [100, 200],
    "learning_rate":  [0.1, 0.05],
    "max_depth":      [2, 3, 5],
    "min_samples_leaf":[10, 50],
    "min_samples_split":[10],
    "max_features":   ["sqrt"],
    "subsample":      [0.8, 1.0],
}

# Cria o modelo base de Gradient Boosting Classifier que será ajustado durante a busca.
gbr = GradientBoostingClassifier(random_state=42)

# Configura o GridSearchCV para testar cada combinação de parâmetros definida acima.
# Usa validação cruzada com 3 folds, avalia pela métrica F1 e executa em paralelo (n_jobs=-1) para acelerar.
grid_search = GridSearchCV(
    estimator=gbr,
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=2
)

# Inicia o processo de busca em grade, treinando e avaliando cada combinação de parâmetros.
print("Iniciando GridSearchCV...")
grid_search.fit(X_train_uf, y_train_uf)

# Exibe no console a melhor combinação de hiperparâmetros encontrada após todos os testes.
print("\nMelhor combinação de hiperparâmetros:")
print(grid_search.best_params_)


Iniciando GridSearchCV...
Fitting 3 folds for each of 48 candidates, totalling 144 fits
[CV] END learning_rate=0.1, max_depth=2, max_features=sqrt, min_samples_leaf=10, min_samples_split=10, n_estimators=100, subsample=0.8; total time=  38.3s
[CV] END learning_rate=0.1, max_depth=2, max_features=sqrt, min_samples_leaf=10, min_samples_split=10, n_estimators=100, subsample=0.8; total time=  39.2s
[CV] END learning_rate=0.1, max_depth=2, max_features=sqrt, min_samples_leaf=10, min_samples_split=10, n_estimators=100, subsample=0.8; total time=  39.8s
[CV] END learning_rate=0.1, max_depth=2, max_features=sqrt, min_samples_leaf=10, min_samples_split=10, n_estimators=100, subsample=1.0; total time=  41.8s
[CV] END learning_rate=0.1, max_depth=2, max_features=sqrt, min_samples_leaf=10, min_samples_split=10, n_estimators=100, subsample=1.0; total time=  42.1s
[CV] END learning_rate=0.1, max_depth=2, max_features=sqrt, min_samples_leaf=10, min_samples_split=10, n_estimators=100, subsample=1.0; t

## **7. Avaliação do Modelo Otimizado**
Nesta etapa, o modelo otimizado é avaliado utilizando o conjunto de teste reservado. São calculadas diversas métricas de desempenho para entender a eficácia do modelo em prever inadimplência.


#### 7.1 O que foi feito
1) Reajuste do modelo com os melhores hiperparâmetros obtidos pela busca:
   - learning_rate: 0.1
   - max_depth: 5
   - max_features: sqrt
   - min_samples_leaf: 50
   - min_samples_split: 10
   - n_estimators: 200
   - subsample: 1.0

2) Geração de previsões para o conjunto de teste.

3) Cálculo das métricas de desempenho e do relatório de classificação.

#### 7.2 Métricas obtidas e o que significam
- Accuracy: 0.9619  
  Proporção de acertos totais. Diz, de forma geral, quantas vezes o modelo acertou entre todos os casos. Em bases desbalanceadas pode ser otimista, por isso olhamos também as métricas por classe.

- Precision: 0.9743  
  Entre todas as previsões de inadimplente (classe positiva), quantas realmente eram inadimplentes. Alta precisão significa poucos falsos positivos (evita classificar como inadimplente quem não é).

- Recall: 0.9392  
  Entre todos os inadimplentes reais, quantos o modelo identificou corretamente. Alto recall significa poucos falsos negativos (evita deixar de identificar inadimplentes).

- F1 score: 0.9564  
  Média harmônica entre precisão e recall. Resume o equilíbrio entre pegar a maior parte dos inadimplentes (recall) sem marcar muitos pagadores como inadimplentes por engano (precisão).

Esses valores indicam um desempenho robusto, com bom equilíbrio entre capturar inadimplentes e evitar alarmes falsos.

#### 7.3 Relatório de classificação (visão por classe)
Mostra precisão, recall e F1 separadamente para cada classe, além do suporte (quantidade de exemplos de cada classe no teste).

- Classe 0 (não inadimplente)  
  precision 0.9527, recall 0.9802, f1 0.9662, support 162839  
  Interpretação: o modelo acerta muito ao reconhecer quem paga (recall alto) e, quando diz que alguém não é inadimplente, costuma estar correto (boa precisão).

- Classe 1 (inadimplente)  
  precision 0.9743, recall 0.9392, f1 0.9564, support 130327  
  Interpretação: o modelo identifica a maioria dos inadimplentes (recall alto) e comete poucos falsos positivos ao rotulá-los (precisão alta), resultando em F1 elevado.

- Visão geral  
  accuracy 0.9619, macro avg f1 0.9613, weighted avg f1 0.9619  
  Interpretação: desempenho consistente entre classes e apropriado para a distribuição real do conjunto de teste.

#### 7.4 O que isso nos diz na prática
- O modelo é eficaz para prever inadimplência, equilibrando bem o risco de perder casos importantes (não detectar inadimplentes) com o custo de marcar pagadores adimplentes como inadimplentes.  
- As métricas elevadas sugerem que as features comportamentais e históricas adicionadas ajudam de fato a separar os padrões das duas classes.  
- Com esses resultados, as previsões podem ser usadas com confiança em análises financeiras, priorização de cobrança e estimativa de recebíveis.


In [25]:
# Reajusta gbr com os melhores hiperparâmetros
gbr = GradientBoostingClassifier(random_state=42, **grid_search.best_params_)
gbr.fit(X_train_uf, y_train_uf)

# Predições e métricas
y_pred_uf = gbr.predict(X_test_uf)

accuracy = accuracy_score(y_test_uf, y_pred_uf)
precision = precision_score(y_test_uf, y_pred_uf, zero_division=0)
recall = recall_score(y_test_uf, y_pred_uf, zero_division=0)
f1 = f1_score(y_test_uf, y_pred_uf, zero_division=0)

print("Métricas de Avaliação no Conjunto de Teste:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nRelatório de Classificação:")
print(classification_report(y_test_uf, y_pred_uf, digits=4))


Métricas de Avaliação no Conjunto de Teste:
Accuracy: 0.9619
Precision: 0.9743
Recall:    0.9392
F1 Score:  0.9564

Relatório de Classificação:
              precision    recall  f1-score   support

           0     0.9527    0.9802    0.9662    162839
           1     0.9743    0.9392    0.9564    130327

    accuracy                         0.9619    293166
   macro avg     0.9635    0.9597    0.9613    293166
weighted avg     0.9623    0.9619    0.9619    293166



## 8. Previsão de Recebíveis em Período Específico

Após o modelo gerar suas previsões, esta etapa transforma os resultados em informações úteis para análise financeira, permitindo calcular o **recebível esperado** para um período específico.  
O objetivo é ajudar a equipe financeira a prever quanto dinheiro deve entrar em caixa em determinada data, já considerando o risco de inadimplência previsto pelo modelo.

### 8.1 Conversão das previsões em DataFrame
- As previsões (`y_pred_uf`) são convertidas em um DataFrame (`y_pred_uf_df`).
- Essa conversão facilita combinar as previsões com os dados originais de teste.

### 8.2 Reanexação de informações ao conjunto de teste
- Cria-se o DataFrame `df_com_predicoes` a partir de `X_test_uf`.
- Adiciona a coluna com as previsões (`yhat_inadimplente`).
- Reinsere as colunas originais `data_vencimento` e `valor_original`, essenciais para análise de fluxo de caixa.

### 8.3 Definição do período de análise
- Cria variáveis `data_inicio` e `data_fim` para definir facilmente o intervalo de datas a ser analisado.
- Permite ajustar o período de forma simples sem alterar o restante do código.

### 8.4 Filtro de registros do período
- Filtra apenas os boletos com data de vencimento dentro do intervalo selecionado.
- Cria o DataFrame `df_periodo`, que contém exclusivamente os registros relevantes para a análise.

### 8.5 Cálculo do recebível esperado
- Cria a coluna `recebivel_esperado`:
  - Define como **0** quando o modelo prevê inadimplência.
  - Mantém o valor original do boleto quando a previsão indica pagamento.
- Soma todos os valores dessa coluna para calcular `recebiveis_periodo`, o total esperado de recebimento no período.

### 8.6 Interpretação prática
Com essa etapa, as previsões do modelo se tornam informações financeiras acionáveis.  
A equipe pode:
- Antecipar o fluxo de caixa esperado em datas futuras.
- Avaliar o impacto da inadimplência prevista.
- Priorizar ações de cobrança ou renegociação para reduzir riscos e garantir liquidez.


In [ ]:
# Converte o vetor de previsões em DataFrame para facilitar junção
y_pred_uf_df = pd.DataFrame(y_pred_uf, index=X_test_uf.index, columns=['yhat_inadimplente'])

# Reanexa as previsões ao conjunto de teste, junto com colunas originais de interesse
df_com_predicoes = X_test_uf.copy()
df_com_predicoes = df_com_predicoes.join(y_pred_uf_df)
df_com_predicoes['data_vencimento'] = df.loc[X_test_uf.index, 'data_vencimento']
df_com_predicoes['valor_original'] = df.loc[X_test_uf.index, 'valor_original']

# Define período de análise (alterável conforme necessidade)
data_inicio = pd.to_datetime("2024-09-18")
data_fim    = pd.to_datetime("2024-09-18")

# Filtra os registros dentro do período selecionado
mask_periodo = (df_com_predicoes['data_vencimento'] >= data_inicio) & \
               (df_com_predicoes['data_vencimento'] <= data_fim)
df_periodo = df_com_predicoes.loc[mask_periodo].copy()

# Calcula o recebível esperado (zera quando previsto como inadimplente)
df_periodo['recebivel_esperado'] = np.where(
    df_periodo['yhat_inadimplente'] == 1,
    0,
    df_periodo['valor_original']
)

# Soma o total de recebíveis esperados no período
recebiveis_periodo = df_periodo['recebivel_esperado'].sum()

print("Total de recebíveis esperados no período:", recebiveis_periodo, "reais")


Total de recebíveis esperados no período: 1591906.81 reais
